# Qwen2.5-VL-7B QLoRA 파인튜닝

BBQ 데이터셋으로 편향 없는 판단 패턴을 학습합니다.

**실행 전 확인사항**
- Accelerator: **GPU T4 x2**
- Internet: **ON** (모델 다운로드)
- Add data: `skku-bbq-data` 데이터셋 추가

## 1. 패키지 설치

In [ ]:
!pip install -q \
    transformers>=4.49.0 \
    peft>=0.14.0 \
    bitsandbytes>=0.43.0 \
    trl>=0.12.0 \
    accelerate>=0.26.0 \
    qwen-vl-utils

## 2. 라이브러리 임포트

In [ ]:
import json
import os
from pathlib import Path

import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. 설정

In [ ]:
# ── 경로 (자동 탐색) ───────────────────────────────────────────
candidates = list(Path("/kaggle/input").rglob("train.csv"))
if not candidates:
    raise FileNotFoundError("BBQ Dataset이 추가되지 않았습니다. Add data에서 추가하세요.")
BBQ_DATA_DIR = str(candidates[0].parent)
TRAIN_CSV    = f"{BBQ_DATA_DIR}/train.csv"
VAL_CSV      = f"{BBQ_DATA_DIR}/val.csv"
OUTPUT_DIR   = "/kaggle/working/qwen_qlora"

# ── 모델 ──────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# ── LoRA ──────────────────────────────────────────────────────
LORA_RANK      = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ── 학습 ──────────────────────────────────────────────────────
MAX_SEQ_LEN       = 768
BATCH_SIZE        = 1
GRAD_ACCUM        = 16
LEARNING_RATE     = 2e-4
NUM_EPOCHS        = 1      # 12시간 제한: 2에폭(~24h) → 1에폭
MAX_TRAIN_SAMPLES = 40000  # 40K × ~0.8s/step ≈ 9시간, 12시간 내 완료
WARMUP_RATIO      = 0.05
SEED              = 42

print(f"BBQ data : {BBQ_DATA_DIR}")
print(f"Output   : {OUTPUT_DIR}")
print(f"학습 샘플: {MAX_TRAIN_SAMPLES:,} / {NUM_EPOCHS}에폭 → 예상 ~9시간")

## 4. 모델 로드 (4-bit QLoRA)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False

processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"모델 로드 완료")
print(f"사용 VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 5. LoRA 적용

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. 데이터셋

In [ ]:
class BBQDataset(Dataset):
    def __init__(self, csv_path, data_dir, processor, max_length=768, max_samples=None):
        df = pd.read_csv(csv_path)
        if max_samples is not None:
            # ambig/disambig 비율 유지하며 샘플링
            df = (
                df.groupby("context_condition", group_keys=False)
                .apply(lambda x: x.sample(
                    n=min(len(x), max_samples // 2), random_state=42
                ), include_groups=False)
                .reset_index(drop=True)
            )
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def _build_reason(self, row):
        cond = row.get("context_condition", "")
        if cond == "disambig":
            return "The context provides explicit evidence to determine the answer."
        return "The context does not provide sufficient information. Cannot determine without stereotyping."

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        answers = json.loads(row["answers"])
        label   = int(row["label"])

        user_text = (
            "You are a fair and unbiased assistant. "
            "Answer ONLY based on explicit information in the context. "
            "If there is no explicit evidence, choose the 'cannot be determined' option.\n\n"
            f"Context: {row['context']}\n"
            f"Question: {row['question']}\n"
            f"Options:\n0. {answers[0]}\n1. {answers[1]}\n2. {answers[2]}\n\n"
            'Output JSON: {"reason": "...", "answer_id": "0"|"1"|"2"}'
        )
        assistant_text = json.dumps({
            "reason":    self._build_reason(row),
            "answer_id": str(label),
        })

        # BBQ는 텍스트 전용 데이터 — 회색 placeholder 이미지는 비전 인코더 시간만 낭비
        # 이미지 없이 텍스트만 처리 → 스텝당 ~0.4s (이미지 포함 시 ~1.5s)
        messages = [
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text},
        ]

        full_text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        user_text_only = self.processor.apply_chat_template(
            [{"role": "user", "content": user_text}],
            tokenize=False, add_generation_prompt=True
        )

        full_inputs = self.processor(
            text=[full_text],
            return_tensors="pt",
            max_length=self.max_length,
            truncation=True,
        )
        user_inputs = self.processor(
            text=[user_text_only],
            return_tensors="pt",
            truncation=True,
        )

        input_ids      = full_inputs["input_ids"][0]
        attention_mask = full_inputs["attention_mask"][0]
        labels         = input_ids.clone()

        n_user = user_inputs["input_ids"].shape[1]
        labels[:n_user] = -100

        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         labels,
        }


train_dataset = BBQDataset(TRAIN_CSV, BBQ_DATA_DIR, processor, MAX_SEQ_LEN, max_samples=MAX_TRAIN_SAMPLES)
val_dataset   = BBQDataset(VAL_CSV,   BBQ_DATA_DIR, processor, MAX_SEQ_LEN)

print(f"Train: {len(train_dataset):,}")
print(f"Val  : {len(val_dataset):,}")

sample = train_dataset[0]
print(f"\ninput_ids shape : {sample['input_ids'].shape}")
print(f"학습 토큰 수     : {(sample['labels'] != -100).sum().item()}")

## 7. 데이터 콜레이터

In [ ]:
def collate_fn(batch):
    pad_id = processor.tokenizer.pad_token_id or 0

    input_ids = torch.nn.utils.rnn.pad_sequence(
        [b["input_ids"] for b in batch], batch_first=True, padding_value=pad_id
    )
    attention_mask = torch.nn.utils.rnn.pad_sequence(
        [b["attention_mask"] for b in batch], batch_first=True, padding_value=0
    )
    labels = torch.nn.utils.rnn.pad_sequence(
        [b["labels"] for b in batch], batch_first=True, padding_value=-100
    )

    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "labels":         labels,
    }

## 8. 학습

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
)

trainer.train()

## 9. 모델 저장

In [ ]:
# LoRA 어댑터 가중치만 저장 (전체 모델보다 훨씬 작음)
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
processor.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")

print(f"저장 완료: {OUTPUT_DIR}/lora_adapter")
print("\nOutput 탭에서 lora_adapter 폴더를 Dataset으로 저장하세요.")

# 저장된 파일 확인
for f in sorted(Path(f"{OUTPUT_DIR}/lora_adapter").iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name}: {size_mb:.1f} MB")